# Week 5: Text Classification

CPS 535, Introduction to Large Language Models

Last week we built `TinyTransformerLM` and trained it to do one thing: given some text, predict the next token. That is a **generative** objective, applied once per position, over the entire vocabulary.

This week the question changes shape. Given a whole piece of text, predict a single label out of a small fixed set of classes, for example "positive" or "negative". That is a **discriminative** objective, applied once per sequence, over a small set of classes.

The good news is that almost nothing about the transformer itself has to change. The two new ideas this week are:

1. How do you turn a sequence of token vectors into a single vector for the whole sequence.
2. Why a classifier does not need a causal mask at all.

Everything else, token embeddings, positional encoding, multi-head self-attention, feedforward layers, residual connections, layer norm, cross-entropy loss, comes forward unchanged from Weeks 3 and 4.

## 1. Generative vs discriminative, side by side

| | Generative (Week 4) | Discriminative (this week) |
|---|---|---|
| Input | partial sequence of tokens | a whole sequence of tokens |
| Output | a distribution over the vocabulary | a distribution over a small set of classes |
| Applied | once per position | once per sequence |
| Label source | the next token in the same text | a human-assigned label (sentiment, topic, spam or not, and so on) |
| Attention direction | causal (can only look at the past) | bidirectional (can look at the whole input) |

Both are trained the same way underneath: a linear layer produces scores (logits) over a set of possible outcomes, softmax turns those into probabilities, and cross-entropy compares the predicted distribution to the true one. That mechanism was fully derived by hand in Weeks 1 and 4, so it is not repeated here. What is new is what sits on top of the transformer, and how the attention pattern changes.

In [2]:
import math, random, collections, io, csv, urllib.request
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import classification_report, confusion_matrix

torch.manual_seed(0)
random.seed(0)
np.random.seed(0)

## 2. From many vectors to one vector: pooling

Every transformer block, from Week 3 onward, takes in one vector per token and returns one vector per token. Classification needs exactly one vector for the whole sequence, since there is only one label to predict. Something has to collapse the sequence of vectors down to a single vector. This step is called **pooling**.

There is a wrinkle first. Training in batches means every sequence in a batch has to be the same length, but real sentences are not the same length. The fix is **padding**: shorter sequences get a special `<pad>` token appended until every sequence in the batch matches the longest one. Just like every other entry in a tokenizer's vocabulary (Week 2), `<pad>` gets its own learned embedding vector, it is not automatically zero.

**Naive pooling**: average every position, padding included. For a sequence of length $T$ with token vectors $h_1, \dots, h_T$:

$$\bar{h} = \frac{1}{T}\sum_{t=1}^{T} h_t$$

**The problem**: the pad vector is not zero and is not meaningful, it just fills space. Averaging it in with the real tokens drags the sequence representation toward a meaningless direction, and the more padding a sequence needs, the worse the distortion gets.

**Masked mean pooling**, the fix: reuse the attention mask $m_t \in \{0, 1\}$ the batch already carries (1 for a real token, 0 for a pad token), sum only the real positions, and divide by the real count instead of $T$:

$$\bar{h} = \frac{\sum_{t=1}^{T} m_t\, h_t}{\sum_{t=1}^{T} m_t}$$

**A worked example.** Take a genuinely negative three-word review, "not really great", and hand-pick toy two-dimensional embeddings for it, plus a `<pad>` embedding (all labelled here as hand-picked, exactly the way Week 3's toy matrices were):

| token | embedding |
|---|---|
| not | $[1.0,\ 0.5]$ |
| really | $[0.2,\ 0.8]$ |
| great | $[0.9,\ -0.3]$ |
| \<pad\> | $[-0.5,\ -0.5]$ |

Batched with one other, longer review, this one needs padding out to length 5, two `<pad>` tokens appended after the three real words. By hand:

In [6]:
not_  = np.array([1.0, 0.5])
really = np.array([0.2, 0.8])
great  = np.array([0.9, -0.3])
pad    = np.array([-0.5, -0.5])

naive_mean_pool  = (not_ + really + great + pad + pad) / 5
masked_mean_pool = (not_ + really + great) / 3

print("naive mean pool  (T=5, 2 pad vectors included):", naive_mean_pool.round(3))
print("masked mean pool (real count=3, pad excluded): ", masked_mean_pool.round(3))

naive mean pool  (T=5, 2 pad vectors included): [0.22 0.  ]
masked mean pool (real count=3, pad excluded):  [0.7   0.333]


The naive pooled vector, $[0.22, 0.00]$, sits nowhere close to any of the three real word embeddings, it has been pulled almost to the origin by the two pad vectors. The masked pooled vector, $[0.70, 0.33]$, is a genuine blend of "not", "really", and "great" only. Same three real words, same padding, very different result, and the gap only grows as more padding is added relative to real content.

## 3. Bidirectional context: dropping the causal mask

Week 4's causal mask stopped each position from attending to positions after it, because generation has to predict a token it has not seen yet, so it must not be allowed to peek at it during training. `torch.tril` built a lower-triangular mask, and Week 4 proved by hand that masked-out positions collapse to an attention weight of exactly zero.

Classification is a different situation entirely. The whole review is available before any decision is made, there is no future token being predicted mid-sequence, so there is nothing to hide. Dropping the mask lets every position attend to every other position, in both directions. This is exactly the difference between decoder-only models like GPT (causal, used for generation) and encoder-only models like BERT (bidirectional, used for tasks like classification), which the textbook's architecture overview touches on.

The softmax-with-mask mechanism itself was already proven by hand in Week 4 (masked positions collapse to zero), so that arithmetic is not repeated. What is worth seeing directly is the practical consequence: the exact same `MultiHeadAttention` class, run on the exact same input, with only the mask toggled off.

In [3]:
class MultiHeadAttention(nn.Module):
    # brought forward from Week 3 / Week 4, unchanged except for the new
    # key_padding_mask argument used to ignore <pad> positions
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x, causal_mask=None, key_padding_mask=None, return_attn=False):
        B, T, _ = x.shape
        Q = self.W_q(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        K = self.W_k(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        V = self.W_v(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_head)
        if causal_mask is not None:
            scores = scores.masked_fill(causal_mask == 0, float('-inf'))
        if key_padding_mask is not None:
            mask = key_padding_mask[:, None, None, :]
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = torch.softmax(scores, dim=-1)
        out = attn @ V
        out = out.transpose(1, 2).contiguous().view(B, T, self.d_model)
        out = self.W_o(out)
        if return_attn:
            return out, attn
        return out

Reuse the "not really great" embeddings from Section 2. "not" is the first word, so under a causal mask it can only ever attend to itself. Remove the mask and it can draw on "really" and "great" too, exactly the context that makes the sentence negative.

In [7]:
not_t, really_t, great_t = torch.tensor(not_), torch.tensor(really), torch.tensor(great)
toy_sentence = torch.stack([not_t, really_t, great_t]).unsqueeze(0).float()  # (1, 3, 2)

toy_mha = MultiHeadAttention(d_model=2, n_heads=1)
causal_mask = torch.tril(torch.ones(3, 3)).bool()

_, attn_causal = toy_mha(toy_sentence, causal_mask=causal_mask, return_attn=True)
_, attn_bidir  = toy_mha(toy_sentence, causal_mask=None, return_attn=True)

print("causal,       'not' attends to [not, really, great]:", attn_causal[0, 0, 0].detach().numpy().round(3))
print("bidirectional, 'not' attends to [not, really, great]:", attn_bidir[0, 0, 0].detach().numpy().round(3))

causal,       'not' attends to [not, really, great]: [1. 0. 0.]
bidirectional, 'not' attends to [not, really, great]: [0.329 0.31  0.361]


Under the causal mask, "not" attends only to itself, weight 1.0 on itself and 0.0 everywhere else, exactly the collapse Week 4 proved. Remove the mask and "not" spreads attention across all three words, including "really" and "great", the two words that come after it and that determine whether the sentence is actually negative. That single change, mask or no mask, is the entire architectural difference between a decoder built for generation and an encoder built for understanding a whole input at once.

## 4. Assembling the classifier

`FeedForward`, the residual connections, and `LayerNorm` are identical to Week 4, so they are brought forward without comment. The one change inside the block: `causal_mask` is never passed, only `key_padding_mask` is, since padding still needs to be ignored even without a causal restriction.

In [11]:
def positional_encoding(seq_len, d_model):
    # brought forward from Week 3 / Week 4, unchanged
    pe = np.zeros((seq_len, d_model))
    position = np.arange(seq_len)[:, None]
    div_term = np.exp(np.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
    pe[:, 0::2] = np.sin(position * div_term)
    pe[:, 1::2] = np.cos(position * div_term)
    return torch.tensor(pe, dtype=torch.float32)

class FeedForward(nn.Module):
    # brought forward from Week 4, unchanged
    def __init__(self, d_model, expansion=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_model * expansion),
            nn.ReLU(),
            nn.Linear(d_model * expansion, d_model),
        )
    def forward(self, x):
        return self.net(x)

class TransformerEncoderBlock(nn.Module):
    # same shape as Week 4's TransformerBlock, causal_mask dropped
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.ff = FeedForward(d_model)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)

    def forward(self, x, key_padding_mask=None):
        x = x + self.attn(self.ln1(x), causal_mask=None, key_padding_mask=key_padding_mask)
        x = x + self.ff(self.ln2(x))
        return x

class TextClassifier(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_layers, max_len, num_classes):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_enc = positional_encoding(max_len, d_model)
        self.blocks = nn.ModuleList(
            [TransformerEncoderBlock(d_model, n_heads) for _ in range(n_layers)]
        )
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, num_classes)

    def forward(self, x, attention_mask):
        B, T = x.shape
        h = self.token_emb(x) + self.pos_enc[:T].to(x.device)
        for block in self.blocks:
            h = block(h, key_padding_mask=attention_mask)
        h = self.ln_f(h)
        mask = attention_mask.unsqueeze(-1).float()
        pooled = (h * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)   # masked mean pooling
        return self.head(pooled)

# shape check on random input before touching real data
m = TextClassifier(vocab_size=100, d_model=32, n_heads=4, n_layers=2, max_len=40, num_classes=2)
x_check = torch.randint(1, 100, (5, 40))
mask_check = torch.ones(5, 40, dtype=torch.long)
print("classifier output shape:", m(x_check, mask_check).shape, "(5 examples, 2 class scores each)")

classifier output shape: torch.Size([5, 2]) (5 examples, 2 class scores each)


## 5. Why accuracy alone can be misleading

Cross-entropy loss for classification is the exact same loss hand-derived in Week 4, just applied once per sequence instead of once per position, so that derivation is not repeated. What genuinely is new this week is how to judge whether a trained classifier is any good.

**Accuracy** (fraction of correct predictions) is the obvious first metric, but it hides a trap on imbalanced data. A dataset that is about 76% positive and 24% negative overall can be misleading. A classifier that always predicts "positive" and never even looks at the text would already score about 76% accuracy, without having learned anything.

**Precision, recall, and F1** fix this by scoring each class separately:
- Precision: of everything predicted positive, what fraction actually was positive.
- Recall: of everything that actually was positive, what fraction did the model catch.
- F1: the harmonic mean of precision and recall, a single number that only stays high if both stay high.

A **confusion matrix** shows the full picture at once: how many of each true class landed in each predicted class.

## 6. Real data: Amazon review sentiment

The dataset is 20,000 short Amazon reviews, each labelled positive or negative, from the UCI Sentiment Labelled Sentences collection (mirrored here as a single CSV on GitHub, `raw.githubusercontent.com`, same fetch pattern as Week 4's Tiny Shakespeare corpus, this step needs internet on first run).

The raw set is imbalanced, about 15,200 positive to 4,800 negative. For today's run the classes are subsampled to a balanced 2,000 and 2,000 (4,000 reviews total) so training is faster and today's numbers are clean and easy to read. A real production system would not throw away 11,000 valid positive examples for convenience, it would instead keep all the data and correct for the imbalance during training (for example, weighting the loss so mistakes on the minority class count for more). That is left for the exercises.

Tokenization here is deliberately crude, lowercase and split on whitespace and punctuation, a small fixed vocabulary of the most frequent words, everything else mapped to `<unk>`. Week 2 built a real BPE tokenizer from scratch, that machinery is not needed again here, the point this week is the classifier, not the tokenizer.

In [8]:
url = "https://raw.githubusercontent.com/pycaret/pycaret/master/datasets/amazon.csv"
raw = urllib.request.urlopen(url, timeout=30).read().decode("utf-8-sig")
rows = list(csv.reader(io.StringIO(raw)))[1:]

pos_reviews = [r[0] for r in rows if r[1] == "1"]
neg_reviews = [r[0] for r in rows if r[1] == "0"]
print(f"full dataset: {len(pos_reviews)} positive, {len(neg_reviews)} negative "
      f"({len(pos_reviews)/(len(pos_reviews)+len(neg_reviews)):.0%} positive)")

full_majority_baseline = len(pos_reviews) / (len(pos_reviews) + len(neg_reviews))
print(f"a classifier that always predicts 'positive' would score {full_majority_baseline:.1%} "
      f"accuracy on the full dataset without reading a single review")

random.shuffle(pos_reviews)
random.shuffle(neg_reviews)
N_PER_CLASS = 2000
texts = pos_reviews[:N_PER_CLASS] + neg_reviews[:N_PER_CLASS]
labels = [1] * N_PER_CLASS + [0] * N_PER_CLASS
combined = list(zip(texts, labels))
random.shuffle(combined)
texts, labels = zip(*combined)
print(f"balanced subsample used today: {len(texts)} reviews")

full dataset: 15233 positive, 4767 negative (76% positive)
a classifier that always predicts 'positive' would score 76.2% accuracy on the full dataset without reading a single review
balanced subsample used today: 4000 reviews


In [9]:
def tokenize(s):
    s = s.lower()
    for ch in ".,!?":
        s = s.replace(ch, f" {ch} ")
    return s.split()

counter = collections.Counter()
for t in texts:
    counter.update(tokenize(t))

VOCAB_SIZE = 5000
word2idx = {"<pad>": 0, "<unk>": 1}
for w, _ in counter.most_common(VOCAB_SIZE - 2):
    word2idx[w] = len(word2idx)
print("vocab size:", len(word2idx))

MAX_LEN = 40
def encode(s):
    ids = [word2idx.get(w, 1) for w in tokenize(s)][:MAX_LEN]
    mask = [1] * len(ids)
    while len(ids) < MAX_LEN:
        ids.append(0)
        mask.append(0)
    return ids, mask

encoded = [encode(t) for t in texts]
X = torch.tensor([e[0] for e in encoded], dtype=torch.long)
M = torch.tensor([e[1] for e in encoded], dtype=torch.long)
Y = torch.tensor(labels, dtype=torch.long)

N_VAL = 800
X_train, X_val = X[:-N_VAL], X[-N_VAL:]
M_train, M_val = M[:-N_VAL], M[-N_VAL:]
Y_train, Y_val = Y[:-N_VAL], Y[-N_VAL:]
print("train:", X_train.shape, " val:", X_val.shape)

val_majority_baseline = max(Y_val.float().mean().item(), 1 - Y_val.float().mean().item())
print(f"majority baseline on today's balanced validation split: {val_majority_baseline:.1%}")

vocab size: 5000
train: torch.Size([3200, 40])  val: torch.Size([800, 40])
majority baseline on today's balanced validation split: 50.1%


## 7. Training

Same optimizer as Week 4, `AdamW`, same cross-entropy loss, just one prediction per sequence now instead of one per position.

In [12]:
model = TextClassifier(vocab_size=len(word2idx), d_model=64, n_heads=4, n_layers=2,
                        max_len=MAX_LEN, num_classes=2)
opt = torch.optim.AdamW(model.parameters(), lr=3e-4)

def batches(X, M, Y, bs=32, shuffle=True):
    idx = list(range(len(X)))
    if shuffle:
        random.shuffle(idx)
    for i in range(0, len(idx), bs):
        b = idx[i:i + bs]
        yield X[b], M[b], Y[b]

def evaluate(X, M, Y):
    model.eval()
    with torch.no_grad():
        logits = model(X, M)
        loss = F.cross_entropy(logits, Y).item()
        preds = logits.argmax(dim=-1)
        acc = (preds == Y).float().mean().item()
    model.train()
    return loss, acc, preds

EPOCHS = 6
for epoch in range(EPOCHS):
    total_loss, n_batches = 0.0, 0
    for xb, mb, yb in batches(X_train, M_train, Y_train):
        opt.zero_grad()
        loss = F.cross_entropy(model(xb, mb), yb)
        loss.backward()
        opt.step()
        total_loss += loss.item()
        n_batches += 1
    val_loss, val_acc, _ = evaluate(X_val, M_val, Y_val)
    print(f"epoch {epoch+1}: train_loss={total_loss/n_batches:.3f}  "
          f"val_loss={val_loss:.3f}  val_acc={val_acc:.3f}")

epoch 1: train_loss=0.668  val_loss=0.616  val_acc=0.683
epoch 2: train_loss=0.526  val_loss=0.453  val_acc=0.789
epoch 3: train_loss=0.434  val_loss=0.420  val_acc=0.810
epoch 4: train_loss=0.385  val_loss=0.408  val_acc=0.824
epoch 5: train_loss=0.347  val_loss=0.398  val_acc=0.812
epoch 6: train_loss=0.313  val_loss=0.388  val_acc=0.829


Validation accuracy climbs well past both the 50% coin-flip baseline and today's 50% balanced majority baseline, from a model that started with completely random weights and has seen only 3,200 labelled reviews. Nothing here was pretrained on outside text, unlike the production systems discussed in Section 9.

## 8. Evaluation: beyond accuracy

In [13]:
_, _, val_preds = evaluate(X_val, M_val, Y_val)
print(classification_report(Y_val.numpy(), val_preds.numpy(), target_names=["negative", "positive"]))
print("confusion matrix (rows = true label, columns = predicted label):")
print(confusion_matrix(Y_val.numpy(), val_preds.numpy()))

              precision    recall  f1-score   support

    negative       0.82      0.83      0.83       399
    positive       0.83      0.82      0.83       401

    accuracy                           0.83       800
   macro avg       0.83      0.83      0.83       800
weighted avg       0.83      0.83      0.83       800

confusion matrix (rows = true label, columns = predicted label):
[[333  66]
 [ 71 330]]


Because today's data was balanced ahead of time, precision, recall, and F1 land close to each other for both classes and close to plain accuracy, there is no imbalance left to hide. Exercise C asks you to rerun this on the original, imbalanced 76 to 24 split and see the metrics pull apart.

## 9. Why not always train from scratch

The classifier just built learns everything, down to basic English word associations, from only 3,200 labelled examples, a tiny amount of text. Real production text classifiers rarely start from nothing. Instead, a transformer is first pretrained on a huge amount of unlabelled text with a self-supervised objective (a bidirectional relative of Week 4's next-token prediction, for example predicting a randomly masked-out word from its context in both directions), and only afterward does a small labelled dataset like this one get used, with exactly the kind of classification head built today attached on top. Sometimes only that head is trained (the pretrained transformer stays frozen), sometimes the whole model is fine-tuned together.

That is the real reason this week's architecture, token embedding, positional encoding, bidirectional self-attention, masked pooling, linear head, is not just a toy shape. It is structurally identical to real text classifiers, the only difference is that today's version started from random weights instead of a checkpoint that had already read billions of words.

## Exercise

**Part A.** Replace masked mean pooling with last-token pooling (the vector at the last real, non-pad position of each sequence). Retrain with the same hyperparameters and compare validation accuracy and F1 to the mean-pooled version above.

**Part B.** Take one very short review and one very long review from the dataset. Using the `great` / `movie` / `pad` style hand-calc from Section 2, compute naive mean pooling and masked mean pooling for each, using the model's own trained embeddings for the real tokens involved. Confirm that the gap between naive and masked pooling grows with the amount of padding.

**Part C.** Rerun data loading with the full, unbalanced dataset (skip the subsampling step) instead of today's balanced 2,000 and 2,000 split. Report precision, recall, and F1 per class, and identify whether the model has gotten noticeably better at one class than the other. Connect what you see back to Section 5.